# 🧠 AI-Based Exam Anxiety Detector
## BERT Fine-Tuning Notebook (Google Colab)

> **Runtime**: `Runtime → Change runtime type → T4 GPU`

---

### Milestones Covered:
- Milestone 1: Environment Setup
- Milestone 2–3: Dataset & Preprocessing
- Milestone 4: BERT Fine-Tuning
- Milestone 5: Evaluation & Visualization

In [ ]:
# ── MILESTONE 1: Install Dependencies ──────────────────────────────
!pip install transformers==4.35.2 datasets evaluate accelerate torch seqeval -q
!pip install scikit-learn matplotlib seaborn plotly -q
print('✅ All packages installed.')

In [ ]:
# ── MILESTONE 2: Dataset Creation ──────────────────────────────────
import pandas as pd
import numpy as np
import re
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

LOW = [
    'I feel pretty calm about the upcoming exam. I studied well.',
    'I am a bit nervous but mostly confident in my preparation.',
    'The exam is tomorrow and I feel ready. Reviewed all my notes.',
    'I studied consistently and feel okay about the test.',
    'I have a mild nervousness but it is manageable.',
    'Looking forward to showing what I have learned in the exam.',
    'I feel prepared and slightly excited about the exam.',
    'A little bit of tension, nothing I cannot handle.',
    'I have done my best to prepare. Feeling neutral about it.',
    'Mild jitters, but I know the material well enough.',
    'I reviewed everything and feel good about tomorrow.',
    'Not too stressed - just a light feeling of anticipation.',
    'I am calm, got enough sleep, and feel ready for the exam.',
    'A small flutter in my stomach but overall feeling fine.',
    'I have everything ready for the exam. Minimal stress.',
    'I trust my preparation. Feeling relatively at ease today.',
    'The exam does not worry me much. I have prepared thoroughly.',
    'I am a bit unsure on two topics but otherwise well prepared.',
    'Slight nervousness is normal, I think. I will do fine.',
    'Reviewed flashcards and feel confident about the content.',
]

MODERATE = [
    'I am somewhat anxious about the exam. Worried I may have missed topics.',
    'The exam tomorrow is making me uneasy. I feel quite unsettled.',
    'I have been stressed about this test. There is a lot riding on it.',
    'My heart races a bit when I think about the exam tomorrow.',
    'I am worried I have not covered everything that might come up.',
    'Feeling tense and distracted. Cannot focus well on revision.',
    'There are parts of the syllabus I am not confident about.',
    'The closer it gets, the more nervous I feel about the exam.',
    'I keep second-guessing my answers in practice tests. Very stressful.',
    'I have been struggling to sleep thinking about the upcoming test.',
    'Mild panic when I cannot recall formulas during revision sessions.',
    'I feel pressure from my family to perform well in the exam.',
    'I keep going over the same notes but still feel uncertain about it.',
    'Anxious thoughts keep interrupting my study sessions lately.',
    'I am worried about running out of time during the exam.',
    'Moderate stress - I know some of it but fear surprises in the exam.',
    'I feel overwhelmed by the volume of material I need to cover.',
    'There is a knot in my stomach every time I think about the exam.',
    'My concentration has been poor because of all this exam stress.',
    'I have been irritable and distracted since exam week began.',
]

HIGH = [
    'I am absolutely terrified about the exam. I cannot stop shaking.',
    'I feel like I am going to fail no matter how much I study.',
    'I cannot eat or sleep properly because of my exam anxiety.',
    'My mind goes completely blank when I try to study. I am panicking.',
    'I am convinced I am going to fail and disappoint everyone.',
    'I feel sick to my stomach every time I think about the exam.',
    'I have been crying every night because of terrible exam stress.',
    'I am so scared I cannot think straight. Nothing is sticking.',
    'I am in complete panic mode. Cannot stop catastrophizing.',
    'The exam has been giving me chest tightness and headaches daily.',
    'I feel hopeless. I do not think I can pass no matter what I do.',
    'My hands tremble whenever I sit down to study for the exam.',
    'I keep having nightmares about failing and freezing in the hall.',
    'Every time I open my books I feel a wave of nausea come over me.',
    'I am spiraling - convinced I know nothing and will fail everything.',
    'I have stopped eating properly because exam anxiety is so overwhelming.',
    'Constant dread. I feel paralyzed and cannot move forward at all.',
    'I feel like I am going to have a breakdown before the exam.',
    'My brain will not retain anything. The pressure is unbearable.',
    'I cannot stop crying. The exam feels impossible and I am terrified.',
]

texts, labels = [], []
for t in LOW:      texts.append(t); labels.append(0)
for t in MODERATE: texts.append(t); labels.append(1)
for t in HIGH:     texts.append(t); labels.append(2)

df = pd.DataFrame({'text': texts, 'label': labels,
                   'label_name': [['Low Anxiety','Moderate Anxiety','High Anxiety'][l] for l in labels]})
print(f'Dataset shape: {df.shape}')
print(df['label_name'].value_counts())
df.head()

In [ ]:
# ── MILESTONE 3: EDA Visualization ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes: ax.set_facecolor('#161b22')

colors = ['#2dd4bf', '#f59e0b', '#f87171']
counts = df['label_name'].value_counts()

bars = axes[0].bar(counts.index, counts.values, color=colors, width=0.55, edgecolor='none')
axes[0].set_title('Anxiety Level Distribution', color='white', fontsize=13)
axes[0].set_ylabel('Count', color='#9ca3af')
axes[0].tick_params(colors='white')
for spine in axes[0].spines.values(): spine.set_visible(False)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, str(val),
                 ha='center', color='white', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, colors=colors, autopct='%1.0f%%',
            textprops={'color':'white'}, wedgeprops={'edgecolor':'#0d1117','linewidth':2})
axes[1].set_title('Label Proportion', color='white', fontsize=13)
plt.suptitle('EDA: Exam Anxiety Dataset', color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('✅ EDA complete')

In [ ]:
# ── MILESTONE 3: Preprocessing ─────────────────────────────────────
def preprocess(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s\.\,\!\?\'"]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['text'] = df['text'].apply(preprocess)
df['text_len'] = df['text'].apply(len)
print(f'Avg text length: {df.text_len.mean():.1f} chars')
df.head(3)

In [ ]:
# ── MILESTONE 4: BERT Training Setup ───────────────────────────────
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AdamW, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

class AnxietyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
X_train, X_val, y_train, y_val = train_test_split(
    df.text.tolist(), df.label.tolist(), test_size=0.2, random_state=42, stratify=df.label)

train_loader = DataLoader(AnxietyDataset(X_train, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader   = DataLoader(AnxietyDataset(X_val, y_val, tokenizer), batch_size=16)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*0.1), total_steps)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Steps: {total_steps}')

In [ ]:
# ── MILESTONE 4: Training Loop ──────────────────────────────────────
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_f1':[]}
best_val_acc = 0.0

from sklearn.metrics import f1_score
import os

for epoch in range(1, 6):
    # Train
    model.train()
    t_loss, correct, total = 0, 0, 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/5 [Train]'):
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=lbls)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        t_loss += out.loss.item()
        preds = torch.argmax(out.logits, dim=1)
        correct += (preds==lbls).sum().item(); total += lbls.size(0)
    history['train_loss'].append(t_loss/len(train_loader))
    history['train_acc'].append(correct/total)
    
    # Eval
    model.eval()
    v_loss, all_p, all_l = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['labels'].to(DEVICE)
            out = model(input_ids=ids, attention_mask=mask, labels=lbls)
            v_loss += out.loss.item()
            all_p.extend(torch.argmax(out.logits,dim=1).cpu().numpy())
            all_l.extend(lbls.cpu().numpy())
    v_acc = accuracy_score(all_l, all_p)
    v_f1  = f1_score(all_l, all_p, average='weighted')
    history['val_loss'].append(v_loss/len(val_loader))
    history['val_acc'].append(v_acc)
    history['val_f1'].append(v_f1)
    
    print(f'Epoch {epoch} | Train Acc: {correct/total:.4f} | Val Acc: {v_acc:.4f} | F1: {v_f1:.4f}')
    
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        os.makedirs('anxiety_bert_model', exist_ok=True)
        model.save_pretrained('anxiety_bert_model')
        tokenizer.save_pretrained('anxiety_bert_model')
        print(f'  ✅ Best model saved (acc={v_acc:.4f})')

print(f'\n🏆 Best Validation Accuracy: {best_val_acc:.4f}')

In [ ]:
# ── MILESTONE 5: Training Curves ────────────────────────────────────
epochs = range(1, 6)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes: ax.set_facecolor('#161b22')

axes[0].plot(epochs, history['train_loss'], 'o-', color='#f59e0b', lw=2.5, label='Train')
axes[0].plot(epochs, history['val_loss'], 's-', color='#f87171', lw=2.5, label='Val')
axes[0].set_title('Loss Curve', color='white'); axes[0].legend(labelcolor='white', frameon=False)

axes[1].plot(epochs, history['train_acc'], 'o-', color='#2dd4bf', lw=2.5, label='Train')
axes[1].plot(epochs, history['val_acc'], 's-', color='#818cf8', lw=2.5, label='Val')
axes[1].set_title('Accuracy Curve', color='white'); axes[1].legend(labelcolor='white', frameon=False)

axes[2].plot(epochs, history['val_f1'], 's-', color='#34d399', lw=2.5, label='Val F1')
axes[2].set_title('F1 Score Curve', color='white'); axes[2].legend(labelcolor='white', frameon=False)

for ax in axes:
    ax.tick_params(colors='white'); ax.set_xlabel('Epoch', color='#9ca3af')
    for s in ax.spines.values(): s.set_visible(False)

plt.suptitle('BERT Training Curves', color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── MILESTONE 5: Confusion Matrix & Report ──────────────────────────
label_names = ['Low Anxiety', 'Moderate Anxiety', 'High Anxiety']
print('=== CLASSIFICATION REPORT ===')
print(classification_report(all_l, all_p, target_names=label_names))

cm = confusion_matrix(all_l, all_p)
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=label_names, yticklabels=label_names,
            linewidths=0.5, linecolor='#0d1117', annot_kws={'size':14,'weight':'bold'}, ax=ax)
ax.set_title('Confusion Matrix', color='white', fontsize=14)
ax.set_xlabel('Predicted', color='#9ca3af'); ax.set_ylabel('True', color='#9ca3af')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Save model to Google Drive (optional) ──────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('anxiety_bert_model', '/content/drive/MyDrive/anxiety_bert_model')
# print('Model saved to Google Drive!')

print('✅ All milestones complete!')
print(f'Best Validation Accuracy: {best_val_acc:.4f}')
print('Model saved in: anxiety_bert_model/')